# LlamaIndex + LangChain Hybrid Agent

**Goal**
- Build a LlamaIndex `VectorStoreIndex` for knowledge
- Expose it as a LangChain Tool
- Add LangChain tools (Calculator, URL summarizer)
- Run a LangChain Agent with conversation memory and concise answers

**Estimated Time:** ~120 minutes

## 1) Setup

In [ ]:
%pip install llama-index-core llama-index-embeddings-openai llama-index-llms-openai langchain langchain-openai openai faiss-cpu python-dotenv requests beautifulsoup4 pydantic

Create a `.env` file:

```env
OPENAI_API_KEY=your_api_key_here
```

## 2) Imports & Initialization

In [ ]:
import os, re, ast, operator as op, requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# LlamaIndex
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import Tool, AgentType, initialize_agent
from langchain.memory import ConversationBufferMemory

# Optional import from prompt (not required below, but included for completeness)
try:
    from langchain.vectorstores import FAISS
except Exception:
    FAISS = None

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

# LLMs
llm_lc = ChatOpenAI(model="gpt-4o-mini", temperature=0)     # LangChain-side
llm_li = LlamaOpenAI(model="gpt-4o-mini", temperature=0)    # LlamaIndex-side
emb_li = OpenAIEmbedding(model="text-embedding-3-small")

## 3) Build LlamaIndex Knowledge Base

In [ ]:
os.makedirs("data", exist_ok=True)
seed = {
  "agentic_ai.txt": "Agentic AI combines LLMs with tools, memory, and goals to act autonomously.",
  "rag.txt": "Retrieval-Augmented Generation grounds answers in external context to reduce hallucinations.",
  "langchain.txt": "LangChain orchestrates prompts, tools, memory, and retrieval.",
  "llamaindex.txt": "LlamaIndex provides data connectors and indexing for LLM apps, enabling flexible RAG pipelines.",
  "langgraph.txt": "LangGraph lets you build explicit stateful graphs (nodes/edges) for agent workflows."
}
for n, t in seed.items():
    with open(os.path.join("data", n), "w", encoding="utf-8") as f:
        f.write(t)

docs = SimpleDirectoryReader("data").load_data()
li_index = VectorStoreIndex.from_documents(docs, embed_model=emb_li)
li_qe = li_index.as_query_engine(llm=llm_li)

print(f"Loaded {len(docs)} docs and built LlamaIndex QueryEngine.")

## 4) Wrap LlamaIndex as a LangChain Tool

In [ ]:
def llamaindex_answer(q: str) -> str:
    """Answer using the indexed docs; include source hints when possible."""
    resp = li_qe.query(q)
    return str(resp)

llamaindex_tool = Tool(
    name="CourseRAG",
    description=(
        "Answer questions from the local knowledge base via LlamaIndex (RAG). "
        "Use for conceptual/framework questions; include short source hints if present."
    ),
    func=llamaindex_answer
)

## 5) Add LangChain Tools (Calculator + URL Summarizer)

In [ ]:
# Safe calculator
OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.Mod: op.mod
}

def _eval(node):
    if isinstance(node, ast.Num):
        return node.n
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.left), _eval(node.right))
    raise ValueError("Disallowed")

def safe_calc(expr: str) -> str:
    cleaned = re.sub(r'[^0-9\+\-\*\/\(\)\.\% ]', '', expr)
    return str(_eval(ast.parse(cleaned, mode="eval").body))

calculator_tool = Tool(
    name="Calculator",
    description="Evaluate arithmetic like 2*(3+4). Return only the number.",
    func=safe_calc
)

# Allowlisted URL summarizer
ALLOWLIST = {"python.org", "openai.com", "lilianweng.github.io"}

def url_summarize(payload: str) -> str:
    m = re.search(r'(https?://\S+)', payload)
    if not m:
        return "Invalid URL."

    target = m.group(1)
    host = re.sub(r"^https?://", "", target).split("/")[0].lower()
    if not any(host.endswith(h) for h in ALLOWLIST):
        return "Blocked: domain not allowlisted."

    r = requests.get(target, timeout=12, headers={"User-Agent": "HybridAgent/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    text = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))[:6000]
    if not text:
        return "No readable content."

    return llm_lc.invoke(f"Summarize in <=120 words, keep key facts:\
\
{text}").content

url_tool = Tool(
    name="UrlSummarizer",
    description="Summarize an allowlisted web page. Input: URL or text containing a URL.",
    func=url_summarize
)

## 6) Memory & System Guidance

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

SYSTEM = (
    "You're a helpful hybrid agent. Prefer CourseRAG for conceptual questions, "
    "Calculator for math, and UrlSummarizer for web pages. "
    "Be concise; cite short hints when using CourseRAG."
)

## 7) Build the LangChain Agent

In [ ]:
tools = [llamaindex_tool, calculator_tool, url_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm_lc,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    agent_kwargs={"system_message": SYSTEM},
)

## 8) Try It

In [ ]:
print("\n--- RAG via LlamaIndex ---")
print(agent.run("In 2 sentences, what is RAG and why is it useful for agents? Include brief source hints."))

print("\n--- Math ---")
print(agent.run("What is (25*4) + 90/3? Return just the number."))

print("\n--- URL (allowlisted) ---")
print(agent.run("Summarize https://python.org in two sentences."))

print("\n--- Memory carryover ---")
print(agent.run("Remember I prefer bullet points."))
print(agent.run("Explain LangGraph briefly, following my preference."))

## 9) Optional: Structured JSON Answers
Add a tool that returns JSON (e.g., `{answer, sources}`) for machine-readable pipelines.

## 10) Optional: Persist LlamaIndex Storage

In [ ]:
li_index.storage_context.persist("li_storage/")

# Later:
# from llama_index.core import load_index_from_storage, StorageContext
# storage = StorageContext.from_defaults(persist_dir="li_storage/")
# li_index = load_index_from_storage(storage)
# li_qe = li_index.as_query_engine(llm=llm_li)

## 11) What to Observe
- Conceptual questions should route to `CourseRAG` (LlamaIndex)
- Arithmetic should use `Calculator`
- Allowlisted web pages should use `UrlSummarizer`
- Conversation memory should adapt style over turns

This scales well: **data/indexing in LlamaIndex**, **orchestration/tools in LangChain**.

## 12) Troubleshooting
- If the agent ignores `CourseRAG`, sharpen tool description ("Answer strictly from indexed docs")
- If URL fetch blocks, add domain to `ALLOWLIST`
- If persistence errors occur, keep embedding model consistent across sessions

You now have a practical hybrid agent: **LlamaIndex for RAG + LangChain for tools/memory/agent control**.